In [ ]:
import os
import openai, json, httpx

client = openai.OpenAI(api_key=os.getenv("OPEN_AI_API_KEY"))

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

messages = [
    {
        "role": "system",
        "content": (
            "You are a movie expert assistant. "
            "You can look up popular movies, get movie details, and find similar movies. "
            "Always use the provided tools to get real data before answering. "
            "When showing movie lists, include the movie ID so users can ask for more details."
        ),
    }
]

In [ ]:
def get_popular_movies():
    response = httpx.get(f"{BASE_URL}/movies")
    return json.dumps(response.json())

def get_movie_details(id):
    response = httpx.get(f"{BASE_URL}/movies/{id}")
    return json.dumps(response.json())

def get_similar_movies(id):
    response = httpx.get(f"{BASE_URL}/movies/{id}/similar")
    return json.dumps(response.json())

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID.",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get a list of movies similar to a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID.",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [13]:
from openai.types.chat import ChatCompletionMessage

def call_ai():
  response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=TOOLS,
  )
  process_ai_response(response.choices[0].message)


def process_ai_response(message: ChatCompletionMessage): 
  if message.tool_calls:
      messages.append({
          "role": "assistant", 
          "content": message.content or "",
          "tool_calls": [
              {
                  "id": tc.id,
                  "type": "function",
                  "function": {
                      "name": tc.function.name,
                      "arguments": tc.function.arguments
                  }
              } 
              for tc in message.tool_calls
          ]
      })
      
      for tool_call in message.tool_calls:
          function_name = tool_call.function.name
          arguments = tool_call.function.arguments

          print(f"Calling function {function_name} with {arguments}")

          try: 
            arguments = json.loads(arguments)
          except json.JSONDecodeError:
            arguments = {}

          function_to_run = FUNCTION_MAP.get(function_name)
          result = function_to_run(**arguments)

          messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result,
          })
      call_ai()

  else:
    messages.append({"role": "assistant", "content": message.content or ""})
    print(f"Assistant: {message.content}")

In [14]:
while True:
    message = input("Send a message to the LLM... ")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: how is the weather of Spain?
Calling function get_weather with {"city":"Spain"}
Assistant: The current weather in Spain is 25°C.


In [15]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant that can use tools to answer questions.'},
 {'role': 'user', 'content': 'how is the weather of Spain?'},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'id': 'call_4LI6pZZf3jcGDyqcXCy8WeZH',
    'type': 'function',
    'function': {'name': 'get_weather', 'arguments': '{"city":"Spain"}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_4LI6pZZf3jcGDyqcXCy8WeZH',
  'name': 'get_weather',
  'content': 'The weather in Spain is 25°C.'},
 {'role': 'assistant', 'content': 'The current weather in Spain is 25°C.'}]